# CRSS and plasticity analysis with `midas-stress`

Everything the package offers for slip-system, Schmid-factor, CRSS activation, yield-proximity, and Taylor-factor analysis. Starts from per-grain stress tensors (from `ms.compute_stress` or equivalent) and walks through the full plasticity toolchain.

## What's inside

1. The slip-system databases — FCC / BCC variants / HCP families.
2. Schmid factor for uniaxial loading.
3. Resolved shear from an arbitrary stress tensor.
4. Dominant & top-K active systems.
5. CRSS activation classification.
6. Yield proximity — which grains yield first?
7. Taylor factor — single-slip polycrystal yield.
8. HCP with family-specific CRSS (basal vs prismatic vs pyramidal).
9. Load-direction sweep and inverse pole figures of Schmid factor.
10. Slip-system rotation primitive (`slip_systems_to_lab`) for custom analyses.

## 1. Setup

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

import midas_stress as ms

plt.rcParams.update({'font.size': 12, 'figure.figsize': (7, 4), 'axes.grid': True, 'grid.alpha': 0.3})

SAVE_DIR = os.path.abspath('.')
RNG = np.random.default_rng(0)

def random_rotations(n, seed):
    rng = np.random.default_rng(seed)
    q = rng.normal(size=(n, 4))
    q /= np.linalg.norm(q, axis=1, keepdims=True)
    w, x, y, z = q.T
    R = np.empty((n, 3, 3))
    R[:,0,0]=1-2*(y*y+z*z); R[:,0,1]=2*(x*y-w*z); R[:,0,2]=2*(x*z+w*y)
    R[:,1,0]=2*(x*y+w*z);   R[:,1,1]=1-2*(x*x+z*z); R[:,1,2]=2*(y*z-w*x)
    R[:,2,0]=2*(x*z-w*y);   R[:,2,1]=2*(y*z+w*x); R[:,2,2]=1-2*(x*x+y*y)
    return R

def save_fig(fig, name):
    p = os.path.join(SAVE_DIR, f'{name}.png')
    fig.savefig(p, bbox_inches='tight', dpi=120)
    print(f'saved: {p}')

print('midas-stress version:', ms.__version__)
print('slip families      :', ms.list_slip_families())

midas-stress version: 0.2.3
slip families      : ['bcc', 'bcc_110', 'bcc_112', 'bcc_123', 'bcc_all', 'fcc', 'fcc_octahedral', 'hcp', 'hcp_basal', 'hcp_prismatic', 'hcp_pyramidal_a', 'hcp_pyramidal_ca']


## 2. Slip-system databases

`get_slip_systems(family)` returns two arrays:
- `n_crystal` (M, 3) — unit plane normals in the crystal frame.
- `b_crystal` (M, 3) — unit slip directions.

All Miller-index / Miller-Bravais reductions are done inside the package — you get Cartesian orthonormal vectors ready for matrix math.

In [2]:
print(f'{"family":<22} {"M":>4}   description')
catalog = [
    ('fcc',             '{111}<110> — 12 octahedral systems'),
    ('bcc_110',         '{110}<111>'),
    ('bcc_112',         '{112}<111>'),
    ('bcc_123',         '{123}<111>'),
    ('bcc',             '{110} ∪ {112} <111>'),
    ('bcc_all',         '{110} ∪ {112} ∪ {123} <111>'),
    ('hcp_basal',       '(0001)<11-20>'),
    ('hcp_prismatic',   '{10-10}<11-20>'),
    ('hcp_pyramidal_a', '{10-11}<11-20>'),
    ('hcp_pyramidal_ca','{11-22}<11-23>'),
    ('hcp',             'basal + prismatic + pyr<a> + pyr<c+a>'),
]
for fam, desc in catalog:
    n, _ = ms.get_slip_systems(fam, c_over_a=ms.HCP_RATIOS['Ti'])
    print(f'{fam:<22} {n.shape[0]:>4}   {desc}')

family                    M   description
fcc                      12   {111}<110> — 12 octahedral systems
bcc_110                  12   {110}<111>
bcc_112                  12   {112}<111>
bcc_123                  24   {123}<111>
bcc                      24   {110} ∪ {112} <111>
bcc_all                  48   {110} ∪ {112} ∪ {123} <111>
hcp_basal                 3   (0001)<11-20>
hcp_prismatic             3   {10-10}<11-20>
hcp_pyramidal_a           6   {10-11}<11-20>
hcp_pyramidal_ca         12   {11-22}<11-23>
hcp                      24   basal + prismatic + pyr<a> + pyr<c+a>


### 2a. Material-aware convenience wrapper

`get_slip_systems_for_material("Cu")` picks a reasonable default family and supplies the HCP c/a when applicable.

In [3]:
for mat in ['Cu', 'Al', 'Ni', 'Fe', 'W', 'Ti']:
    n, b = ms.get_slip_systems_for_material(mat)
    print(f'{mat:<4} auto-selected: {n.shape[0]} systems')

Cu   auto-selected: 12 systems
Al   auto-selected: 12 systems
Ni   auto-selected: 12 systems
Fe   auto-selected: 24 systems
W    auto-selected: 24 systems
Ti   auto-selected: 24 systems


## 3. Schmid factor — uniaxial loading

For unit load direction $\hat\ell$, per-grain Schmid factors are

$$m_s = (\hat n_{\text{lab}} \cdot \hat\ell)(\hat b_{\text{lab}}\cdot\hat\ell).$$

Bounded $\le 0.5$ — the classical maximum occurs when $\hat n$ and $\hat b$ are both at 45° to the load.

In [4]:
grains = ms.read_grains(ms.example_data_path('GrainsSim.csv'))
orient = grains['orientations']
vols   = (4 / 3) * np.pi * grains['radii']**3
N      = orient.shape[0]

n_fcc, b_fcc = ms.get_slip_systems('fcc')

m_z = ms.schmid_factor(orient, [0, 0, 1], n_fcc, b_fcc)   # (N, 12)
m_x = ms.schmid_factor(orient, [1, 0, 0], n_fcc, b_fcc)

fig, ax = plt.subplots()
ax.hist(m_z.max(axis=1), bins=30, alpha=0.6, label='max Schmid (load z)')
ax.hist(m_x.max(axis=1), bins=30, alpha=0.6, label='max Schmid (load x)')
ax.axvline(0.5, color='k', ls='--', label='theoretical ceiling (0.5)')
ax.set_xlabel('max Schmid factor per grain')
ax.set_title('FCC Schmid distribution for two load directions')
ax.legend()
fig.tight_layout()
save_fig(fig, 'schmid_distribution')
plt.close(fig)

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/schmid_distribution.png


## 4. Resolved shear from a full stress tensor

`resolved_shear_stress(stress, orient, n, b)` is the general case — works for *any* stress state per grain (per-grain heterogeneity, biaxial / shear loads, post-equilibrium stresses).

$$\tau_s = \hat n_{\text{lab}}^\top\,\sigma\,\hat b_{\text{lab}}.$$

In [5]:
# Synthetic strains + compute_stress produces realistic per-grain stress tensors.
eps_macro = np.diag([-3e-4, -3e-4, 1e-3])
eps_scatter = 2e-4 * RNG.normal(size=(N, 3, 3))
eps_scatter = 0.5 * (eps_scatter + eps_scatter.swapaxes(-1, -2))
strains = eps_macro + eps_scatter

result = ms.compute_stress(
    strain=strains, stiffness=ms.get_stiffness('Cu'),
    orient=orient, volumes=vols,
    applied_stress=np.diag([0.0, 0.0, 0.150]),   # uniaxial 150 MPa z
)
stress = result['stress_corrected']

tau = ms.resolved_shear_stress(stress, orient, n_fcc, b_fcc)  # (N, 12)
print(f'|tau| max per grain: mean = {np.abs(tau).max(axis=1).mean()*1e3:.2f} MPa')
print(f'|tau| max overall  : {np.abs(tau).max()*1e3:.2f} MPa')

fig, ax = plt.subplots()
ax.boxplot(np.abs(tau) * 1e3, showfliers=False)
ax.set_xlabel('system index')
ax.set_ylabel(r'$|\tau|$ [MPa]')
ax.set_title('per-system resolved shear distribution (FCC, 150 MPa along z)')
fig.tight_layout()
save_fig(fig, 'resolved_shear_boxplot')
plt.close(fig)

|tau| max per grain: mean = 62.71 MPa
|tau| max overall  : 96.16 MPa


saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/resolved_shear_boxplot.png


## 5. Dominant and top-K systems

`dominant_slip_system(...)` returns the ranked-by-|score| systems for each grain. Pass either a load direction (Schmid ranking) or a stress tensor (resolved-shear ranking).

In [6]:
dom = ms.dominant_slip_system(orient, n_fcc, b_fcc, stress=stress, top_k=3)

print('keys:', list(dom.keys()))
print(f'best_index shape: {dom["best_index"].shape}   (one system index per grain)')
print(f'rank shape      : {dom["rank"].shape}         (top-3)')
print(f'top_score shape : {dom["top_score"].shape}')

# Distribution of the #1 system across all grains
counts = np.bincount(dom['best_index'], minlength=n_fcc.shape[0])
fig, ax = plt.subplots()
ax.bar(range(n_fcc.shape[0]), counts, color='teal')
ax.set_xlabel('slip system index (FCC)')
ax.set_ylabel('# grains where it is #1')
ax.set_title('dominant system across the polycrystal')
fig.tight_layout()
save_fig(fig, 'dominant_system')
plt.close(fig)

keys: ['score', 'signed', 'rank', 'top_score', 'best_index', 'best_score']
best_index shape: (250,)   (one system index per grain)
rank shape      : (250, 3)         (top-3)
top_score shape : (250, 3)
saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/dominant_system.png


## 6. CRSS classification — `active_systems_from_crss`

A system is *active* if $|\tau_s| \ge \tau_c^{(s)}$. Pass CRSS as either a scalar (same threshold for every system) or an array of per-system thresholds (useful for HCP).

In [7]:
crss_scalar = 0.025   # 25 MPa, in the same units as stress (GPa)
act = ms.active_systems_from_crss(stress, orient, n_fcc, b_fcc, crss=crss_scalar)

print(f'fraction of grains with any active system: {act["fraction_grains_yielding"] * 100:.1f} %')
print(f'mean active systems per yielding grain   : {act["n_active_per_grain"][act["grains_any_active"]].mean():.2f}')

fig, ax = plt.subplots()
ax.bar(range(n_fcc.shape[0]), act['fraction_active_per_system'] * 100, color='steelblue')
ax.set_xlabel('slip system index')
ax.set_ylabel('% of grains active')
ax.set_title(f'FCC activation at CRSS = {crss_scalar*1e3:.0f} MPa')
fig.tight_layout()
save_fig(fig, 'crss_activation_per_system')
plt.close(fig)

fraction of grains with any active system: 100.0 %
mean active systems per yielding grain   : 6.36


saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/crss_activation_per_system.png


### 6a. CRSS sweep — yield fraction vs threshold

In [8]:
crss_list = np.linspace(0.005, 0.10, 20)  # 5 MPa ... 100 MPa
frac = []
for c in crss_list:
    r = ms.active_systems_from_crss(stress, orient, n_fcc, b_fcc, crss=c)
    frac.append(r['fraction_grains_yielding'])

fig, ax = plt.subplots()
ax.plot(crss_list * 1e3, np.array(frac) * 100, 'o-')
ax.set_xlabel('CRSS [MPa]')
ax.set_ylabel(r'% grains with $\geq 1$ active system')
ax.set_title('yield-onset spectrum for the polycrystal')
fig.tight_layout()
save_fig(fig, 'crss_sweep')
plt.close(fig)

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/crss_sweep.png


## 7. Yield proximity — which grain yields first?

Rank grains by $\max_s |\tau_s|/\tau_c^{(s)}$. Grains with a score $\ge 1$ are yielding; the ranking identifies the first-to-yield grains even when most of the sample is still elastic.

In [9]:
prox = ms.yield_proximity(stress, orient, n_fcc, b_fcc, crss=crss_scalar)

fig, ax = plt.subplots()
ax.hist(prox['proximity'], bins=40, color='firebrick', edgecolor='k')
ax.axvline(1.0, color='k', ls='--', label='yield threshold')
ax.set_xlabel(r'$\max_s |\tau_s|/\tau_c$')
ax.set_ylabel('grain count')
ax.set_title('yield-proximity distribution')
ax.legend()
fig.tight_layout()
save_fig(fig, 'yield_proximity_hist')
plt.close(fig)

# Most-yielded grains
print('Top 5 most-stressed grains:')
print(f'  {"rank":<5}{"grain":<8}{"proximity":<12}{"critical_system":<16}')
for rank, g in enumerate(prox['grains_sorted'][:5], 1):
    print(f'  {rank:<5}{g:<8}{prox["proximity"][g]:<12.3f}{prox["critical_system"][g]:<16}')

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/yield_proximity_hist.png
Top 5 most-stressed grains:
  rank grain   proximity   critical_system 
  1    245     3.846       7               
  2    26      3.835       9               
  3    201     3.675       6               
  4    70      3.606       8               
  5    159     3.437       4               


### 7a. Spatial map of yield proximity

Color each grain by its proximity score, at its centroid.

In [10]:
pos = grains['positions']
fig, ax = plt.subplots()
sc = ax.scatter(pos[:, 0], pos[:, 1], c=prox['proximity'],
                s=grains['radii'] / 5, cmap='plasma')
ax.set_xlabel('x [um]')
ax.set_ylabel('y [um]')
ax.set_aspect('equal')
plt.colorbar(sc, ax=ax, label='yield proximity')
ax.set_title('where the sample yields first')
fig.tight_layout()
save_fig(fig, 'yield_proximity_map')
plt.close(fig)

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/yield_proximity_map.png


## 8. Taylor factor — polycrystal yield estimate

Single-slip upper bound: $M_g = 1/\max_s |m_s|$ per grain; volume-weighted mean for the polycrystal. Estimated yield stress is $M_{\text{poly}} \cdot \tau_{\text{CRSS}}$.

In [11]:
taylor = ms.taylor_factor(orient, [0, 0, 1], n_fcc, b_fcc, volumes=vols)
print(f'polycrystal M (volume-weighted): {taylor["M_poly"]:.3f}')
print(f'polycrystal M (equal-weighted) : {taylor["M_uniform"]:.3f}')
print(f'yield stress = M x CRSS        : {taylor["M_poly"] * crss_scalar * 1e3:.1f} MPa')

fig, ax = plt.subplots()
ax.hist(taylor['M_per_grain'], bins=40, color='slateblue', edgecolor='k')
ax.axvline(taylor['M_poly'], color='r', ls='--', label=f"M_poly = {taylor['M_poly']:.2f}")
ax.set_xlabel('per-grain Taylor factor')
ax.set_ylabel('grain count')
ax.set_title('Taylor-factor distribution (FCC, load = z)')
ax.legend()
fig.tight_layout()
save_fig(fig, 'taylor_factor_hist')
plt.close(fig)

polycrystal M (volume-weighted): 2.225
polycrystal M (equal-weighted) : 2.225
yield stress = M x CRSS        : 55.6 MPa


saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/taylor_factor_hist.png


## 9. HCP with family-specific CRSS (Ti example)

Real HCP metals have very different CRSS for each family (basal is softest, pyramidal is hardest). Pass an *array* of CRSS values to `active_systems_from_crss` / `yield_proximity` — one per system.

In [12]:
N_ti = 500
orient_ti = random_rotations(N_ti, seed=5)
vols_ti   = np.ones(N_ti)
stress_ti = np.broadcast_to(np.diag([0.0, 0.0, 0.250]), (N_ti, 3, 3)).copy()

# Build a combined HCP system list with per-system CRSS
c_a = ms.HCP_RATIOS['Ti']
segments = [
    ('hcp_basal',        0.025),
    ('hcp_prismatic',    0.060),
    ('hcp_pyramidal_a',  0.080),
    ('hcp_pyramidal_ca', 0.180),
]
n_parts, b_parts, crss_parts = [], [], []
labels_per_system = []
for fam, tau_c in segments:
    n, b = ms.get_slip_systems(fam, c_over_a=c_a)
    n_parts.append(n); b_parts.append(b)
    crss_parts.append(np.full(n.shape[0], tau_c))
    labels_per_system.extend([fam] * n.shape[0])
n_all    = np.concatenate(n_parts, axis=0)
b_all    = np.concatenate(b_parts, axis=0)
crss_arr = np.concatenate(crss_parts)
print(f'total systems : {n_all.shape[0]}')

act = ms.active_systems_from_crss(stress_ti, orient_ti, n_all, b_all, crss=crss_arr)

# Per-family activity
fams = [s[0] for s in segments]
for fam, tau_c in segments:
    mask = np.array([l == fam for l in labels_per_system])
    active_any_in_family = act['active'][:, mask].any(axis=1)
    print(f'{fam:<22} CRSS={tau_c*1e3:<4.0f} MPa   yielding grains: {active_any_in_family.mean()*100:.1f} %')

total systems : 24
hcp_basal              CRSS=25   MPa   yielding grains: 87.6 %
hcp_prismatic          CRSS=60   MPa   yielding grains: 69.6 %
hcp_pyramidal_a        CRSS=80   MPa   yielding grains: 57.0 %
hcp_pyramidal_ca       CRSS=180  MPa   yielding grains: 0.0 %


## 10. Load-direction sweep — IPF of max Schmid factor

The Schmid-factor max over the FCC 12 systems depends on loading direction. Sample the unit sphere in the lab frame, report the max Schmid averaged over the polycrystal.

In [13]:
n_dirs = 200
rng = np.random.default_rng(11)
v = rng.normal(size=(n_dirs, 3))
v /= np.linalg.norm(v, axis=1, keepdims=True)

mean_max = np.zeros(n_dirs)
for i, d in enumerate(v):
    mean_max[i] = ms.schmid_factor(orient, d, n_fcc, b_fcc).max(axis=1).mean()

fig, ax = plt.subplots()
sc = ax.scatter(v[:, 0], v[:, 1], c=mean_max, cmap='viridis', s=20)
ax.set_xlabel('lab $x$')
ax.set_ylabel('lab $y$')
ax.set_aspect('equal')
plt.colorbar(sc, ax=ax, label='mean max Schmid')
ax.set_title('load-direction sweep, projected onto lab xy plane')
fig.tight_layout()
save_fig(fig, 'schmid_load_sweep')
plt.close(fig)

best = v[np.argmax(mean_max)]
worst = v[np.argmin(mean_max)]
print(f'softest load dir (highest mean Schmid): {best.round(3)}   m_max = {mean_max.max():.3f}')
print(f'hardest load dir (lowest mean Schmid) : {worst.round(3)}  m_max = {mean_max.min():.3f}')

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/schmid_load_sweep.png
softest load dir (highest mean Schmid): [ 0.126 -0.821 -0.557]   m_max = 0.457
hardest load dir (lowest mean Schmid) : [-0.703 -0.266  0.66 ]  m_max = 0.445


## 11. `slip_systems_to_lab` — the low-level primitive

If you need the plane normals / slip directions *themselves* in the lab frame (e.g. to overlay on an experiment, compute higher-order kinematic tensors, etc.), `slip_systems_to_lab` returns them.

In [14]:
n_lab, b_lab = ms.slip_systems_to_lab(orient[:5], n_fcc, b_fcc)
print(f'n_lab shape: {n_lab.shape}   b_lab shape: {b_lab.shape}')
print(f'(5 grains, 12 FCC systems, 3-vector each)')
print()
# Sanity: each is still a unit vector after rotation
print(f'||n_lab|| (should all be 1): '
      f'{np.linalg.norm(n_lab, axis=-1).min():.6f} ... {np.linalg.norm(n_lab, axis=-1).max():.6f}')
print(f'n_lab . b_lab (should all be 0, slip direction lies in plane): '
      f'max |dot| = {np.abs((n_lab * b_lab).sum(axis=-1)).max():.2e}')

n_lab shape: (5, 12, 3)   b_lab shape: (5, 12, 3)
(5 grains, 12 FCC systems, 3-vector each)

||n_lab|| (should all be 1): 0.999999 ... 1.000001
n_lab . b_lab (should all be 0, slip direction lies in plane): max |dot| = 7.80e-07


## 12. Comparing slip-family hierarchies — BCC

BCC iron has three candidate families — `{110}`, `{112}`, and `{123}` planes all accommodate $\langle 111\rangle$ slip. `midas-stress` lets you pick the family (or combine them) and see how the yielding grains change.

In [15]:
N_fe = 500
orient_fe = random_rotations(N_fe, seed=23)
stress_fe = np.broadcast_to(np.diag([0.0, 0.0, 0.120]), (N_fe, 3, 3)).copy()

for fam in ['bcc_110', 'bcc_112', 'bcc_123', 'bcc', 'bcc_all']:
    n, b = ms.get_slip_systems(fam)
    r = ms.active_systems_from_crss(stress_fe, orient_fe, n, b, crss=0.050)
    print(f'{fam:<10} {n.shape[0]:>4} systems   yielding grains: {r["fraction_grains_yielding"]*100:.1f} %')

bcc_110      12 systems   yielding grains: 83.2 %
bcc_112      12 systems   yielding grains: 85.8 %
bcc_123      24 systems   yielding grains: 88.8 %
bcc          24 systems   yielding grains: 88.4 %
bcc_all      48 systems   yielding grains: 89.0 %


## 13. End-to-end plasticity pipeline from a MIDAS CSV

In [16]:
# 1) Read grains
grains = ms.read_grains(ms.example_data_path('GrainsSim.csv'))
N = grains['orientations'].shape[0]

# 2) Optional frame conversion MIDAS -> APS sample
sam = ms.grains_midas_to_sample(
    grains['orientations'], grains['positions'],
    strains, target_frame='aps',
)

# 3) Compute stresses (with equilibrium correction)
result = ms.compute_stress(
    strain=sam['strains'],
    stiffness=ms.get_stiffness('Cu'),
    orient=sam['orientations'],
    volumes=(4/3) * np.pi * grains['radii']**3,
    confidences=grains['confidences'],
    min_confidence=0.5,
)
stress = result['stress_corrected']

# 4) Plasticity analysis
n_sys, b_sys = ms.get_slip_systems_for_material('Cu')
crss = 0.025
act  = ms.active_systems_from_crss(stress, sam['orientations'], n_sys, b_sys, crss=crss)
prox = ms.yield_proximity(stress, sam['orientations'], n_sys, b_sys, crss=crss)
taylor = ms.taylor_factor(sam['orientations'], [0, 0, 1], n_sys, b_sys,
                          volumes=(4/3)*np.pi*grains['radii']**3)

print(f'polycrystal mean von Mises : {result["von_mises"].mean()*1e3:.1f} MPa')
print(f'fraction grains yielding   : {act["fraction_grains_yielding"]*100:.1f} %')
print(f'Taylor factor (poly)       : {taylor["M_poly"]:.3f}')
print(f'predicted yield stress     : {taylor["M_poly"] * crss * 1e3:.1f} MPa')

polycrystal mean von Mises : 89.7 MPa
fraction grains yielding   : 92.0 %
Taylor factor (poly)       : 2.214
predicted yield stress     : 55.4 MPa


## 14. Cheat sheet — which function when?

| Question | Function |
|---|---|
| Best-oriented grain for uniaxial load? | `schmid_factor(...).max(axis=1)` |
| Shear on a specific slip system under arbitrary loading? | `resolved_shear_stress(stress, ...)` |
| Top-K most-stressed systems per grain? | `dominant_slip_system(..., top_k=K)` |
| Which grains have $\ge 1$ active system? | `active_systems_from_crss(...)` |
| Ranking grains by proximity to yield? | `yield_proximity(...)` |
| Polycrystal yield-stress estimate? | `taylor_factor(...)` + CRSS |
| HCP with family-specific CRSS? | build per-system CRSS array, pass to above |
| Need normals / directions in lab frame? | `slip_systems_to_lab(...)` |

## 15. Gotchas

- Units: CRSS **must** be in the same units as the stress tensor (GPa or MPa — pick one and stick with it).
- Schmid factor is always $\in [0, 0.5]$ for uniaxial loading. Larger values → bug (non-unit load direction? transposed `orient`?).
- `resolved_shear_stress` handles arbitrary stress states — do *not* pass it a scalar load direction; use `schmid_factor` for that.
- HCP systems require `c_over_a`. Use `HCP_RATIOS[name]` or pass explicitly.
- Grain `orient` must be crystal→lab. If your pipeline produces lab→crystal (a common inconsistency), transpose before calling anything here.